In [2]:
import pandas as pd
df = pd.read_csv("cleaned_data/filter.tsv", sep="\t")
print("Total:", len(df))
print(df["score"].describe())
print(df.duplicated(subset=["kazakh", "russian"]).sum(), "duplicates")


Total: 5183424
count    5.183424e+06
mean     7.616522e-01
std      2.462976e-01
min     -4.333480e-01
25%      6.362847e-01
50%      8.773167e-01
75%      9.426097e-01
max      1.000000e+00
Name: score, dtype: float64
4757627 duplicates


In [ ]:
df_unique = df.drop_duplicates(subset=["kazakh", "russian"])

# df_unique.to_csv("cleaned_data/sample_700k.tsv", sep="\t", index=False)

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load scored dataset
filter_df = pd.read_csv("cleaned_data/filter.tsv", sep="\t")  # has kazakh, russian, score

# Load raw train/test (without scores)
train_df = pd.read_csv("cleaned_data/train.txt", sep="\t", names=["kazakh", "russian"], on_bad_lines="skip")
test_df = pd.read_csv("cleaned_data/test.txt", sep="\t", names=["kazakh", "russian"], on_bad_lines="skip")

print(f"📦 Original train size: {len(train_df)}")
print(f"📦 Original test size:  {len(test_df)}")

# Combine and merge with scores
combined_df = pd.concat([train_df, test_df], ignore_index=True)
scored_df = combined_df.merge(filter_df, on=["kazakh", "russian"], how="left")

# Drop unmatched
missing = scored_df["score"].isna().sum()
if missing > 0:
    print(f"⚠️ Dropping {missing} pairs not found in filter.tsv")
    scored_df = scored_df.dropna(subset=["score"])

# Drop duplicates
dedup_df = scored_df.drop_duplicates(subset=["kazakh", "russian"])
print(f"✅ Deduplicated total: {len(dedup_df)}")

# Resplit
train_new, test_new = train_test_split(dedup_df, test_size=0.2, random_state=42)

# Save
train_new.to_csv("cleaned_data/train_dedup.txt", sep="\t", index=False)
test_new.to_csv("cleaned_data/test_dedup.txt", sep="\t", index=False)

print("✅ Saved deduplicated train and test splits with scores.")


📦 Original train size: 4127735
📦 Original test size:  1031877
⚠️ Dropping 27611 pairs not found in filter.tsv
✅ Deduplicated total: 423483
✅ Saved deduplicated train and test splits with scores.


In [5]:
from transformers import MarianTokenizer

model_id = "Helsinki-NLP/opus-mt-tc-big-kk-ru"
tokenizer = MarianTokenizer.from_pretrained(model_id)


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OSError: Helsinki-NLP/opus-mt-tc-big-kk-ru is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`